# Phase 1 — Eastern Cape Historical Election Pipeline (2000–2021)

This notebook rebuilds the Eastern Cape Local Government Election history directly from the repository's **raw IEC files**.

### Phase 1 outputs
- Municipality-election panel (turnout and competition)
- Long-format party vote/share table
- Wide-format selected-party share table
- Municipality crosswalk used for historical boundary harmonisation
- Validation report against the existing cleaned prototype outputs

### Methodological choices
- Eastern Cape only
- PR ballot only for consistent party-support measurement
- Municipality boundaries harmonised to the 33-code analytical geography used by the project
- Registered voters and spoilt ballots are counted once per voting district
- Party votes are aggregated from party rows
- ENP is calculated from the full raw party distribution before any party grouping
- Existing `clean_res*.csv` and `clean_competition_all.csv` are used only as validation references, not as source data


In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / "data" / "raw" / "detailed").exists():
            return p
    raise FileNotFoundError(
        "Could not locate project root. Run this notebook from inside "
        "Dirisa_TeamQualification or one of its subfolders."
    )

PROJECT_ROOT = find_project_root()
RAW_DETAILED = PROJECT_ROOT / "data" / "raw" / "detailed"
PROCESSED_ELECTIONS = PROJECT_ROOT / "data" / "processed" / "elections"
PHASE1_DIR = PROCESSED_ELECTIONS / "phase1"
PHASE1_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    2000: RAW_DETAILED / "2000 LGE.csv",
    2006: RAW_DETAILED / "2006 LGE.csv",
    2011: RAW_DETAILED / "2011_detailed" / "EC.csv",
    2016: RAW_DETAILED / "2016_detailed" / "EC.csv",
    2021: RAW_DETAILED / "2021_detailed" / "EC.csv",
}

print("Project root:", PROJECT_ROOT)
for year, path in FILES.items():
    print(year, "->", path.relative_to(PROJECT_ROOT), "| exists:", path.exists())
assert all(p.exists() for p in FILES.values()), "One or more raw election files are missing."


Project root: /mnt/data/dirisa_phase1/Dirisa_TeamQualification
2000 -> data/raw/detailed/2000 LGE.csv | exists: True
2006 -> data/raw/detailed/2006 LGE.csv | exists: True
2011 -> data/raw/detailed/2011_detailed/EC.csv | exists: True
2016 -> data/raw/detailed/2016_detailed/EC.csv | exists: True
2021 -> data/raw/detailed/2021_detailed/EC.csv | exists: True


## 1. Load the two IEC schema generations

The repository contains two layouts:

- **2000/2006:** older IEC layout (`Voting District`, `Ballot Type`, `Valid Votes Cast`, etc.)
- **2011/2016/2021:** newer layout (`VotingDistrict`, `BallotType`, `TotalValidVotes`, etc.)

The 2011 EC file is UTF-16 encoded, while the other files use different encodings.


In [2]:
def read_election_file(year, path):
    encodings = ["utf-16"] if year == 2011 else ["utf-8-sig", "latin1"]
    last_error = None
    for enc in encodings:
        try:
            df = pd.read_csv(path, encoding=enc, low_memory=False)
            df.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in df.columns]
            print(f"{year}: {df.shape[0]:,} rows × {df.shape[1]} cols | encoding={enc}")
            return df
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error

raw = {year: read_election_file(year, path) for year, path in FILES.items()}

schema_summary = pd.DataFrame({
    year: pd.Series(df.columns) for year, df in raw.items()
})
display(schema_summary)


2000: 199,566 rows × 12 cols | encoding=latin1


2006: 374,695 rows × 14 cols | encoding=latin1
2011: 76,488 rows × 11 cols | encoding=utf-16


2016: 102,919 rows × 11 cols | encoding=utf-8-sig
2021: 156,083 rows × 11 cols | encoding=utf-8-sig


,2000,2006,2011,2016,2021
0,Electoral Event,Electoral Event,Province,Province,Province
1,Province,Province,Municipality,Municipality,Municipality
2,Municipality,Municipality,Ward,Ward,Ward
3,Ward,Ward,VotingDistrict,VotingDistrict,VotingDistrict
4,Voting District,Voting District,VotingStationName,VotingStationName,VotingStationName
5,Party,Party,RegisteredVoters,RegisteredVoters,RegisteredVoters
6,Ballot Type,Ballot Type,BallotType,BallotType,BallotType
7,Registered Voters,Registered Voters,SpoiltVotes,SpoiltVotes,SpoiltVotes
8,% Voter Turnout,% Voter Turnout,PartyName,PartyName,PartyName
9,Total Votes Cast,MEC7 Votes,TotalValidVotes,TotalValidVotes,TotalValidVotes


## 2. Municipality harmonisation

Historical municipality codes changed because of mergers and boundary changes.  
The mapping below converts historical codes to the project's current 33-code analytical geography.

**Important:** this is an analytical harmonisation crosswalk. It should remain documented because boundary harmonisation is a substantive modelling assumption.


In [3]:
CODE_MAP = {
    "EC103": "EC101",
    "EC107": "EC101",
    "EC125": "BUF",
    "EC127": "EC129",
    "EC128": "EC129",
    "EC132": "EC139",
    "EC133": "EC139",
    "EC134": "EC139",
    "EC143": "EC145",
    "EC144": "EC145",
    "EC151": "EC443",
    "EC152": "EC444",
    "EC05B2": "EC442",
    "EC05B3": "EC441",
}

TARGET_CODES = {
    "BUF", "NMA",
    "EC101", "EC102", "EC104", "EC105", "EC106", "EC108", "EC109",
    "EC121", "EC122", "EC123", "EC124", "EC126", "EC129",
    "EC131", "EC135", "EC136", "EC137", "EC138", "EC139",
    "EC141", "EC142", "EC145",
    "EC153", "EC154", "EC155", "EC156", "EC157",
    "EC441", "EC442", "EC443", "EC444",
}

def extract_historical_code(value):
    text = str(value).strip()
    upper = text.upper()

    # Older 2000 data labels Nelson Mandela Bay by city rather than NMA.
    if upper.startswith("PORT ELIZABETH"):
        return "NMA"

    code = text.split(" - ")[0].strip().upper()

    # District Management Areas are not treated as present-day local municipalities.
    if code.startswith("ECDMA"):
        return None

    return code

def harmonise_code(value):
    code = extract_historical_code(value)
    if code is None:
        return None
    return CODE_MAP.get(code, code)

crosswalk = pd.DataFrame(
    [{"HistoricalCode": k, "TargetCode": v} for k, v in sorted(CODE_MAP.items())]
)
crosswalk.to_csv(PHASE1_DIR / "municipality_crosswalk.csv", index=False)
display(crosswalk)


,HistoricalCode,TargetCode
0,EC05B2,EC442
1,EC05B3,EC441
2,EC103,EC101
3,EC107,EC101
4,EC125,BUF
5,EC127,EC129
6,EC128,EC129
7,EC132,EC139
8,EC133,EC139
9,EC134,EC139


## 3. Standardise, filter to Eastern Cape + PR, and aggregate

For the older files, `Valid Votes Cast` is the **party-row vote count**; `Total Votes Cast`, registered voters and spoilt votes repeat across party rows for the same voting district.

For the newer files, `TotalValidVotes` is the **party-row vote count**.

Therefore:
1. voting-district facts are deduplicated before summing registered voters/spoilt ballots;
2. party votes are summed from all party rows;
3. valid votes are the sum of party votes;
4. votes cast = valid votes + spoilt votes;
5. turnout = votes cast / registered voters.


In [4]:
def numeric(series):
    return pd.to_numeric(
        series.astype(str)
              .str.replace(",", "", regex=False)
              .str.replace("%", "", regex=False)
              .str.strip(),
        errors="coerce"
    )

def normalise_party_name(value):
    name = str(value).strip()
    upper = name.upper()

    if upper == "AFRICAN NATIONAL CONGRESS":
        return "ANC"
    if "DEMOCRATIC ALLIANCE" in upper or "DEMOKRATIESE ALLIANSIE" in upper:
        return "DA"
    if upper == "ECONOMIC FREEDOM FIGHTERS":
        return "EFF"
    if upper == "UNITED DEMOCRATIC MOVEMENT":
        return "UDM"
    if upper == "AFRICAN TRANSFORMATION MOVEMENT":
        return "ATM"

    return name

def process_election(year, df):
    x = df.copy()

    # Eastern Cape only
    x = x[
        x["Province"].astype(str).str.strip().str.casefold().eq("eastern cape")
    ].copy()

    if year <= 2006:
        ballot_col = "Ballot Type"
        vd_col = "Voting District"
        party_col = "Party"

        x = x[x[ballot_col].astype(str).str.strip().str.upper().eq("PR")].copy()
        x["RegisteredVoters"] = numeric(x["Registered Voters"])
        x["SpoiltVotes"] = numeric(x["Spoilt Votes"])
        x["PartyVotes"] = numeric(x["Valid Votes Cast"])
    else:
        ballot_col = "BallotType"
        vd_col = "VotingDistrict"
        party_col = "PartyName"

        x = x[x[ballot_col].astype(str).str.strip().str.upper().eq("PR")].copy()
        x["RegisteredVoters"] = numeric(x["RegisteredVoters"])
        x["SpoiltVotes"] = numeric(x["SpoiltVotes"])
        x["PartyVotes"] = numeric(x["TotalValidVotes"])

    x["HistoricalCode"] = x["Municipality"].map(extract_historical_code)
    x["MuniCode"] = x["Municipality"].map(harmonise_code)
    x["Party"] = x[party_col].map(normalise_party_name)

    # Keep only the harmonised analytical geography.
    dropped = (
        x.loc[~x["MuniCode"].isin(TARGET_CODES),
              ["Municipality", "HistoricalCode"]]
         .drop_duplicates()
         .sort_values(["HistoricalCode", "Municipality"], na_position="last")
    )

    x = x[x["MuniCode"].isin(TARGET_CODES)].copy()

    # Registered voters and spoilt ballots repeat on each party row.
    district_facts = x.drop_duplicates(["MuniCode", vd_col])

    municipality = (
        district_facts.groupby("MuniCode", as_index=False)
        .agg(
            RegisteredVoters=("RegisteredVoters", "sum"),
            SpoiltVotes=("SpoiltVotes", "sum"),
            VotingDistricts=(vd_col, "nunique"),
        )
    )

    # Party votes are genuinely row-level and must be summed.
    party_votes = (
        x.groupby(["MuniCode", "Party"], as_index=False)
         .agg(PartyVotes=("PartyVotes", "sum"))
    )

    valid = (
        party_votes.groupby("MuniCode", as_index=False)
        .agg(ValidVotes=("PartyVotes", "sum"))
    )

    municipality = municipality.merge(valid, on="MuniCode", how="left")
    municipality["VotesCast"] = municipality["ValidVotes"] + municipality["SpoiltVotes"]
    municipality["Turnout_%"] = (
        100 * municipality["VotesCast"] / municipality["RegisteredVoters"]
    )
    municipality["Year"] = year

    party_votes = party_votes.merge(valid, on="MuniCode", how="left")
    party_votes["VoteShare_%"] = (
        100 * party_votes["PartyVotes"] / party_votes["ValidVotes"]
    )
    party_votes["Year"] = year

    # Competition is calculated from the complete party distribution.
    competition_rows = []
    for code, g in party_votes.groupby("MuniCode"):
        shares = np.sort(g["VoteShare_%"].to_numpy(dtype=float))[::-1]
        competition_rows.append({
            "MuniCode": code,
            "TopShare_%": shares[0],
            "Margin_pts": shares[0] - shares[1] if len(shares) > 1 else shares[0],
            "ENP": 1 / np.sum((shares / 100) ** 2),
            "NumberOfParties": len(shares),
            "Year": year,
        })

    competition = pd.DataFrame(competition_rows)

    return municipality, party_votes, competition, dropped

results = {}
for year, df in raw.items():
    results[year] = process_election(year, df)
    muni, party, comp, dropped = results[year]
    print(
        f"{year}: {len(muni)} municipalities | "
        f"{int(muni['RegisteredVoters'].sum()):,} registered | "
        f"{int(muni['ValidVotes'].sum()):,} valid votes"
    )


2000: 32 municipalities | 2,494,398 registered | 1,366,265 valid votes


2006: 33 municipalities | 2,905,636 registered | 1,596,953 valid votes
2011: 33 municipalities | 3,111,535 registered | 1,754,042 valid votes


2016: 33 municipalities | 3,337,532 registered | 1,832,048 valid votes


2021: 33 municipalities | 3,253,307 registered | 1,471,069 valid votes


## 4. Build the five-election analytical tables

In [5]:
municipality_panel = pd.concat(
    [results[y][0] for y in sorted(results)],
    ignore_index=True
).sort_values(["MuniCode", "Year"]).reset_index(drop=True)

party_long = pd.concat(
    [results[y][1] for y in sorted(results)],
    ignore_index=True
).sort_values(["Year", "MuniCode", "Party"]).reset_index(drop=True)

competition = pd.concat(
    [results[y][2] for y in sorted(results)],
    ignore_index=True
).sort_values(["MuniCode", "Year"]).reset_index(drop=True)

municipality_panel = municipality_panel.merge(
    competition,
    on=["MuniCode", "Year"],
    how="left",
    validate="one_to_one"
)

# Historical lag features — no future information is used.
municipality_panel["PreviousRegisteredVoters"] = (
    municipality_panel.groupby("MuniCode")["RegisteredVoters"].shift(1)
)
municipality_panel["PreviousTurnout_%"] = (
    municipality_panel.groupby("MuniCode")["Turnout_%"].shift(1)
)
municipality_panel["PreviousENP"] = (
    municipality_panel.groupby("MuniCode")["ENP"].shift(1)
)
municipality_panel["PreviousMargin_pts"] = (
    municipality_panel.groupby("MuniCode")["Margin_pts"].shift(1)
)
municipality_panel["PreviousTopShare_%"] = (
    municipality_panel.groupby("MuniCode")["TopShare_%"].shift(1)
)

municipality_panel["TurnoutChange_pts"] = (
    municipality_panel["Turnout_%"] - municipality_panel["PreviousTurnout_%"]
)
municipality_panel["ENPChange"] = (
    municipality_panel["ENP"] - municipality_panel["PreviousENP"]
)
municipality_panel["MarginChange_pts"] = (
    municipality_panel["Margin_pts"] - municipality_panel["PreviousMargin_pts"]
)

display(municipality_panel.head(10))
print("Municipality-election observations:", len(municipality_panel))
print("Expected: 32 (2000) + 33×4 =", 32 + 33*4)


,MuniCode,RegisteredVoters,SpoiltVotes,VotingDistricts,ValidVotes,VotesCast,Turnout_%,Year,TopShare_%,Margin_pts,ENP,NumberOfParties,PreviousRegisteredVoters,PreviousTurnout_%,PreviousENP,PreviousMargin_pts,PreviousTopShare_%,TurnoutChange_pts,ENPChange,MarginChange_pts
0,BUF,341665,4073,216,194108,198181,58.004478,2000,81.267130,68.015229,1.472733,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BUF,358460,2761,298,184573,187334,52.260782,2006,81.334215,69.401808,1.477063,9,341665.0,58.004478,1.472733,68.015229,81.267130,-5.743696,0.004330,1.386580
2,BUF,384910,4016,308,212540,216556,56.261464,2011,69.125341,48.864684,1.916003,8,358460.0,52.260782,1.477063,69.401808,81.334215,4.000681,0.438940,-20.537124
3,BUF,419044,5843,339,227772,233615,55.749516,2016,59.864250,36.412729,2.369499,12,384910.0,56.261464,1.916003,48.864684,69.125341,-0.511948,0.453495,-12.451956
4,BUF,406486,3571,351,180265,183836,45.225666,2021,60.510914,41.017946,2.381045,26,419044.0,55.749516,2.369499,36.412729,59.864250,-10.523850,0.011547,4.605217
5,EC101,28769,422,28,17328,17750,61.698356,2000,55.459372,22.212604,2.355619,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,EC101,34211,341,33,18020,18361,53.669872,2006,62.280799,33.518313,2.112625,5,28769.0,61.698356,2.355619,22.212604,55.459372,-8.028484,-0.242994,11.305709
7,EC101,38214,291,42,23537,23828,62.354111,2011,54.335727,9.835578,2.027128,5,34211.0,53.669872,2.112625,33.518313,62.280799,8.684239,-0.085496,-23.682735
8,EC101,40887,359,46,24709,25068,61.310441,2016,51.021895,4.617751,2.100601,4,38214.0,62.354111,2.027128,9.835578,54.335727,-1.043670,0.073473,-5.217827
9,EC101,39913,272,46,18433,18705,46.864430,2021,46.628330,6.862692,2.632657,8,40887.0,61.310441,2.100601,4.617751,51.021895,-14.446011,0.532056,2.244941


Municipality-election observations: 164
Expected: 32 (2000) + 33×4 = 164


## 5. Selected-party wide table

The complete long table remains the authoritative party dataset.  
This wide table is only a convenient modelling/EDA view for major parties present in the historical data.


In [6]:
selected_parties = ["ANC", "DA", "EFF", "UDM", "ATM"]

party_wide = (
    party_long[party_long["Party"].isin(selected_parties)]
    .pivot_table(
        index=["Year", "MuniCode"],
        columns="Party",
        values="VoteShare_%",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

party_wide.columns.name = None
for party in selected_parties:
    if party not in party_wide.columns:
        party_wide[party] = 0.0

party_wide = party_wide[
    ["Year", "MuniCode", *selected_parties]
].sort_values(["MuniCode", "Year"]).reset_index(drop=True)

display(party_wide.head(10))


,Year,MuniCode,ANC,DA,EFF,UDM,ATM
0,2000,BUF,81.267130,13.251901,0.000000,1.509984,0.000000
1,2006,BUF,81.334215,11.932406,0.000000,1.383193,0.000000
2,2011,BUF,69.125341,20.260657,0.000000,0.735391,0.000000
3,2016,BUF,59.864250,23.451522,8.206891,1.292521,0.000000
4,2021,BUF,60.510914,19.492969,12.371231,1.103930,0.889246
5,2000,EC101,55.459372,33.246768,0.000000,5.257387,0.000000
6,2006,EC101,62.280799,28.762486,0.000000,0.000000,0.000000
7,2011,EC101,54.335727,44.500149,0.000000,0.000000,0.000000
8,2016,EC101,51.021895,46.404144,1.857623,0.000000,0.000000
9,2021,EC101,46.628330,39.765638,4.123040,0.000000,0.146476


## 6. Validation against existing cleaned prototype outputs

The existing clean files are not used to construct the new data.  
They are loaded **afterward** to verify that the raw-data reconstruction reproduces the established totals and competition measures.


In [7]:
validation_rows = []

for year in sorted(results):
    suffix = str(year)[-2:]
    old_path = PROCESSED_ELECTIONS / f"clean_res{suffix}.csv"

    if old_path.exists():
        old = pd.read_csv(old_path)
        new = results[year][0]

        check = new.merge(
            old[["MuniCode", "RegisteredVoters", "ValidVotes", "Turnout_%"]],
            on="MuniCode",
            suffixes=("_new", "_old"),
            how="inner"
        )

        validation_rows.append({
            "Year": year,
            "NewMunicipalities": len(new),
            "ReferenceMunicipalities": len(old),
            "MatchedMunicipalities": len(check),
            "MaxRegisteredVoterDifference": (
                check["RegisteredVoters_new"] - check["RegisteredVoters_old"]
            ).abs().max(),
            "MaxValidVoteDifference": (
                check["ValidVotes_new"] - check["ValidVotes_old"]
            ).abs().max(),
            "MaxTurnoutDifference_pts": (
                check["Turnout_%_new"] - check["Turnout_%_old"]
            ).abs().max(),
        })

validation = pd.DataFrame(validation_rows)
display(validation)

competition_reference = PROCESSED_ELECTIONS / "clean_competition_all.csv"
if competition_reference.exists():
    old_comp = pd.read_csv(competition_reference)
    comp_check = competition.merge(
        old_comp,
        on=["MuniCode", "Year"],
        suffixes=("_new", "_old"),
        how="inner"
    )
    print("Competition observations matched:", len(comp_check))
    for col in ["TopShare_%", "Margin_pts", "ENP"]:
        max_diff = (
            comp_check[f"{col}_new"] - comp_check[f"{col}_old"]
        ).abs().max()
        print(f"Max {col} difference:", max_diff)


,Year,NewMunicipalities,ReferenceMunicipalities,MatchedMunicipalities,MaxRegisteredVoterDifference,MaxValidVoteDifference,MaxTurnoutDifference_pts
0,2000,32,32,32,0,0,1.421085e-14
1,2006,33,33,33,0,0,7.105427e-15
2,2011,33,33,33,0,0,1.421085e-14
3,2016,33,33,33,0,0,1.421085e-14
4,2021,33,33,33,0,0,1.421085e-14


Competition observations matched: 164
Max TopShare_% difference: 1.4210854715202004e-14
Max Margin_pts difference: 1.4210854715202004e-14
Max ENP difference: 1.3322676295501878e-15


## 7. Quality-control gates

In [8]:
issues = []

# Unique municipality-year rows
duplicate_count = municipality_panel.duplicated(["MuniCode", "Year"]).sum()
issues.append(("Duplicate municipality-year rows", int(duplicate_count)))

# Turnout must be plausible.
bad_turnout = (~municipality_panel["Turnout_%"].between(0, 100)).sum()
issues.append(("Turnout outside 0–100", int(bad_turnout)))

# ENP cannot be below 1.
bad_enp = (municipality_panel["ENP"] < 1).sum()
issues.append(("ENP below 1", int(bad_enp)))

# Margin and top share are percentages.
bad_margin = (~municipality_panel["Margin_pts"].between(0, 100)).sum()
bad_top = (~municipality_panel["TopShare_%"].between(0, 100)).sum()
issues.append(("Margin outside 0–100", int(bad_margin)))
issues.append(("Top share outside 0–100", int(bad_top)))

# Party shares should sum to ~100 within each municipality-election.
share_sums = (
    party_long.groupby(["Year", "MuniCode"])["VoteShare_%"].sum()
)
bad_share_sums = (~np.isclose(share_sums, 100, atol=1e-8)).sum()
issues.append(("Party-share groups not summing to 100", int(bad_share_sums)))

qc = pd.DataFrame(issues, columns=["Check", "ProblemRows"])
display(qc)

assert qc["ProblemRows"].sum() == 0, "Phase 1 quality-control gate failed."
assert len(municipality_panel) == 164, "Unexpected municipality-election observation count."

print("PASS: Phase 1 core quality-control checks.")


,Check,ProblemRows
0,Duplicate municipality-year rows,0
1,Turnout outside 0–100,0
2,ENP below 1,0
3,Margin outside 0–100,0
4,Top share outside 0–100,0
5,Party-share groups not summing to 100,0


PASS: Phase 1 core quality-control checks.


## 8. Save final Phase 1 outputs

In [9]:
output_files = {
    "municipality_election_panel_2000_2021.csv": municipality_panel,
    "party_shares_long_2000_2021.csv": party_long,
    "party_shares_selected_wide_2000_2021.csv": party_wide,
    "competition_metrics_2000_2021.csv": competition,
    "phase1_validation.csv": validation,
    "phase1_quality_control.csv": qc,
}

for filename, df in output_files.items():
    path = PHASE1_DIR / filename
    df.to_csv(path, index=False)
    print("Saved:", path.relative_to(PROJECT_ROOT), "|", len(df), "rows")

print("\nPhase 1 complete.")


Saved: data/processed/elections/phase1/municipality_election_panel_2000_2021.csv | 164 rows
Saved: data/processed/elections/phase1/party_shares_long_2000_2021.csv | 1030 rows
Saved: data/processed/elections/phase1/party_shares_selected_wide_2000_2021.csv | 164 rows
Saved: data/processed/elections/phase1/competition_metrics_2000_2021.csv | 164 rows
Saved: data/processed/elections/phase1/phase1_validation.csv | 5 rows
Saved: data/processed/elections/phase1/phase1_quality_control.csv | 6 rows

Phase 1 complete.


## Phase 1 completion note

The election pipeline is now separated cleanly from the earlier prototype:

- raw IEC files are the source of truth;
- the old clean files are validation references only;
- historical municipality codes are explicitly harmonised;
- turnout and competition are reconstructed for all five elections;
- party-level information is preserved in long format;
- lag features are created without using future election information.

### Boundary caveat
The crosswalk harmonises historical municipalities to the project's 33-code analytical geography. This supports longitudinal modelling, but historical boundary changes mean the resulting observations should not be interpreted as perfectly unchanged geographic units through time. This limitation should be documented in the final report.

### Next phase
Phase 2 should build the demographic dataset with explicit temporal alignment rather than attaching the same Census snapshot to every election year.
